## Combine OOF + test predictions from 02 (baseline), 03 (CatBoost), 04 (XGBoost), 05 (LightGBM) into a single weighted-average ensemble.

Weights are optimized on the OOF predictions (all models share the
exact same folds from 07_Cross_Validation.py, so this is a valid,
leak-free comparison) and then applied to the test predictions.

In [3]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

sys.path.insert(0, BASE_DIR)

fe = importlib.import_module("06_Feature_Engineering")

ARTIFACT_DIR = "./artifacts"
SUBMISSION_PATH = "./submission.csv"

# name -> (oof_path, test_pred_path)
MODELS = {
    "baseline_logreg": (
        f"{ARTIFACT_DIR}/oof_baseline_logreg.npy",
        f"{ARTIFACT_DIR}/test_pred_baseline_logreg.csv",
    ),
    "catboost": (
        f"{ARTIFACT_DIR}/oof_catboost.npy",
        f"{ARTIFACT_DIR}/test_pred_catboost.csv",
    ),
    "xgboost": (
        f"{ARTIFACT_DIR}/oof_xgboost.npy",
        f"{ARTIFACT_DIR}/test_pred_xgboost.csv",
    ),
    "lightgbm": (
        f"{ARTIFACT_DIR}/oof_lightgbm.npy",
        f"{ARTIFACT_DIR}/test_pred_lightgbm.csv",
    ),
}

In [4]:
def load_available_models():
    """Only include models whose artifact files actually exist on disk."""
    available = {}
    for name, (oof_path, test_path) in MODELS.items():
        if os.path.exists(oof_path) and os.path.exists(test_path):
            available[name] = (oof_path, test_path)
        else:
            print(f"[skip] {name}: missing {oof_path} or {test_path} "
                  f"(run its training script first)")
    if len(available) < 2:
        raise RuntimeError(
            "Need at least 2 trained models with saved OOF/test predictions "
            "to build an ensemble. Run 02-05 first."
        )
    return available

In [5]:
def optimize_weights(oof_matrix, y_true):
    """
    Find non-negative weights (summing to 1) over OOF predictions that
    maximize ROC-AUC, via SLSQP on the negative AUC.
    """
    n_models = oof_matrix.shape[1]
    init_w = np.full(n_models, 1.0 / n_models)

    def neg_auc(w):
        w = np.clip(w, 0, None)
        w = w / (w.sum() + 1e-12)
        blend = oof_matrix @ w
        return -roc_auc_score(y_true, blend)

    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1},)
    bounds = [(0.0, 1.0)] * n_models

    result = minimize(
        neg_auc, init_w, method="SLSQP", bounds=bounds, constraints=constraints,
        options={"maxiter": 200},
    )
    weights = np.clip(result.x, 0, None)
    weights = weights / weights.sum()
    return weights

In [6]:
def main():
    available = load_available_models()
    names = list(available.keys())

    # ---- load OOF predictions + ground truth ----
    train, _ = fe.load_raw_data()
    y_true = train[fe.TARGET].values

    oof_matrix = np.column_stack([np.load(available[n][0]) for n in names])

    print("=" * 70)
    print("INDIVIDUAL MODEL OOF PERFORMANCE")
    print("=" * 70)
    for i, name in enumerate(names):
        auc = roc_auc_score(y_true, oof_matrix[:, i])
        print(f"{name:>16s}: OOF AUC = {auc:.5f}")

    # ---- simple average baseline ensemble ----
    simple_avg = oof_matrix.mean(axis=1)
    simple_auc = roc_auc_score(y_true, simple_avg)
    print(f"\n{'simple_average':>16s}: OOF AUC = {simple_auc:.5f}")

    # ---- optimized weighted ensemble ----
    weights = optimize_weights(oof_matrix, y_true)
    weighted_oof = oof_matrix @ weights
    weighted_auc = roc_auc_score(y_true, weighted_oof)

    print("\n" + "=" * 70)
    print("OPTIMIZED WEIGHTS")
    print("=" * 70)
    for name, w in zip(names, weights):
        print(f"{name:>16s}: weight = {w:.4f}")
    print(f"\n{'weighted_ensemble':>16s}: OOF AUC = {weighted_auc:.5f}")

    # Use whichever is better on OOF: optimized weights or plain average
    # (guards against overfitting the weights when models are very close).
    if weighted_auc >= simple_auc:
        final_weights = weights
        print("\n-> Using optimized weights for the final submission.")
    else:
        final_weights = np.full(len(names), 1.0 / len(names))
        print("\n-> Optimized weights didn't beat simple average; "
              "using equal weights instead.")

    # ---- build final test prediction ----
    test_preds = []
    id_col_values = None
    for name in names:
        df = pd.read_csv(available[name][1])
        if id_col_values is None:
            id_col_values = df[fe.ID_COL].values
        test_preds.append(df[fe.TARGET].values)
    test_matrix = np.column_stack(test_preds)

    final_test_pred = test_matrix @ final_weights

    submission = pd.DataFrame({fe.ID_COL: id_col_values, fe.TARGET: final_test_pred})
    submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\nSaved final submission -> {SUBMISSION_PATH}")
    print(submission.head())

In [7]:
if __name__ == "__main__":
    main()

[skip] lightgbm: missing ./artifacts/oof_lightgbm.npy or ./artifacts/test_pred_lightgbm.csv (run its training script first)
INDIVIDUAL MODEL OOF PERFORMANCE
 baseline_logreg: OOF AUC = 0.93820
        catboost: OOF AUC = 0.94146
         xgboost: OOF AUC = 0.94140

  simple_average: OOF AUC = 0.94109

OPTIMIZED WEIGHTS
 baseline_logreg: weight = 0.3310
        catboost: weight = 0.3307
         xgboost: weight = 0.3383

weighted_ensemble: OOF AUC = 0.94110

-> Using optimized weights for the final submission.

Saved final submission -> ./submission.csv
       id  Will_Buy_EV
0  668665     0.013134
1  668666     0.062598
2  668667     0.011210
3  668668     0.005978
4  668669     0.035470
